# 01 — Data Understanding

Purpose: inspect the usage dataset and IAM risk catalogue before cleaning or feature engineering.
This notebook does not train the final model.

In [1]:
# AI-Based IAM Permission Optimizer — Model V2
# Run notebooks in order: 01 → 10
# Raw CSVs should be available in the project root or adjust RAW_DIR below.
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

# Works when notebook is opened from notebooks/ OR project root
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USAGE_PATH = RAW_DIR / "ml_ready_synthetic.csv"
RISK_PATH = RAW_DIR / "AWS_Risk_Weight_Table_FINAL_4007_Actions_Shuffled.csv"

if not USAGE_PATH.exists():
    raise FileNotFoundError(f"Usage dataset not found: {USAGE_PATH}")
if not RISK_PATH.exists():
    raise FileNotFoundError(f"Risk catalogue not found: {RISK_PATH}")

usage = pd.read_csv(USAGE_PATH)
risk = pd.read_csv(RISK_PATH)

print("Usage shape:", usage.shape)
print("Risk catalogue shape:", risk.shape)

Usage shape: (20000, 25)
Risk catalogue shape: (4005, 6)


In [2]:
print("\nUsage columns:")
print(usage.columns.tolist())

print("\nRisk columns:")
print(risk.columns.tolist())

print("\nUsage dtypes:")
display(usage.dtypes.to_frame("dtype"))


Usage columns:
['id', 'user_id', 'role_id', 'position', 'department', 'team', 'action', 'resource', 'resource_scope', 'permission_age_days', 'policy_attachment', 'usage_count', 'unique_days_used', 'days_since_last_use', 'first_used', 'last_used', 'success_count', 'failure_count', 'success_rate', 'failure_rate', 'permission_status', 'service', 'operation_type', 'risk_level', 'risk_weight']

Risk columns:
['action', 'service', 'operation_type', 'risk_level', 'risk_weight', 'reason']

Usage dtypes:


,dtype
id,int64
user_id,object
role_id,object
position,object
department,object
team,object
action,object
resource,object
resource_scope,object
permission_age_days,int64


In [3]:
print("Missing values — usage:")
display(usage.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("Duplicate complete rows:", usage.duplicated().sum())
print("Unique users:", usage["user_id"].nunique())
print("Unique roles:", usage["role_id"].nunique())
print("Unique actions used:", usage["action"].nunique())
print("Unique risk actions:", risk["action"].nunique())

Missing values — usage:


,missing
last_used,2671
first_used,2671
role_id,0
user_id,0
department,0
team,0
action,0
position,0
id,0
resource_scope,0


Duplicate complete rows: 0
Unique users: 100
Unique roles: 20
Unique actions used: 3882
Unique risk actions: 4005


In [4]:
if "permission_status" in usage.columns:
    print("Permission label distribution:")
    display(usage["permission_status"].value_counts(dropna=False))

    print("\nExcessive rate by usage_count (label-generation audit):")
    audit = (
        usage.groupby("usage_count")["permission_status"]
        .agg(samples="count", unique_labels="nunique")
        .reset_index()
    )
    audit["excessive_rate"] = (
        usage.assign(_excessive=usage["permission_status"].eq("EXCESSIVE"))
        .groupby("usage_count")[_excessive if False else "permission_status"]
        .size()
    ) if False else np.nan
    audit = usage.groupby("usage_count").agg(
        samples=("permission_status", "size"),
        unique_labels=("permission_status", "nunique"),
        excessive_rate=("permission_status", lambda s: (s == "EXCESSIVE").mean())
    ).reset_index()
    display(audit.head(30))

Permission label distribution:


permission_status
INTENDED     16000
EXCESSIVE     4000
Name: count, dtype: int64


Excessive rate by usage_count (label-generation audit):


,usage_count,samples,unique_labels,excessive_rate
0,0,2671,2,0.524523
1,1,3418,2,0.326214
2,2,1272,2,0.182390
3,3,1007,2,0.179742
4,4,860,2,0.143023
5,5,753,2,0.148738
6,6,671,2,0.162444
7,7,540,2,0.116667
8,8,455,2,0.134066
9,9,424,2,0.110849


In [5]:
# Basic mathematical consistency checks
for c in ["usage_count", "unique_days_used", "days_since_last_use", "success_count", "failure_count"]:
    if c in usage.columns:
        usage[c] = pd.to_numeric(usage[c], errors="coerce")

count_mismatch = (
    usage["success_count"].fillna(0) + usage["failure_count"].fillna(0)
    != usage["usage_count"].fillna(0)
).sum()
print("success_count + failure_count != usage_count:", int(count_mismatch))

# Risk catalogue conflicts
risk_conflicts = risk.groupby("action")[["service", "operation_type", "risk_level", "risk_weight"]].nunique().gt(1).any(axis=1).sum()
print("Actions with conflicting risk metadata:", int(risk_conflicts))

success_count + failure_count != usage_count: 0
Actions with conflicting risk metadata: 0


In [6]:
summary = {
    "usage_rows": int(len(usage)),
    "usage_columns": int(len(usage.columns)),
    "users": int(usage["user_id"].nunique()),
    "roles": int(usage["role_id"].nunique()),
    "unique_actions_used": int(usage["action"].nunique()),
    "risk_catalog_actions": int(risk["action"].nunique()),
    "duplicate_rows": int(usage.duplicated().sum()),
    "missing_cells": int(usage.isna().sum().sum()),
    "count_mismatch_rows": int(count_mismatch),
    "risk_catalog_conflicting_actions": int(risk_conflicts)
}

with open(EVAL_DIR / "data_understanding_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:", EVAL_DIR / "data_understanding_summary.json")


Saved: C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\outputs\data_understanding_summary.json
